# 🚀 05. 비동기 백그라운드 태스크 및 SSE 푸시 스트리밍

이 노트북은 FastAPI 백엔드에서 문헌 매트릭스 합성 등 대규모 비동기 백그라운드 태스크를 기동하고 실행 상태를 추적하는 원리와 완료 시 클라이언트에 실시간 알림을 보내주는 SSE (Server-Sent Events) 브로드캐스팅, 그리고 30분 만료 보안 세션 및 물리 파일을 정리하는 데몬의 백그라운드 작동 구조를 시뮬레이션으로 학습합니다.

### ⚙️ 필요한 라이브러리 및 모듈 임포트
비동기 큐(`Queue`) 관리와 데몬 작동 타임스탬프 계산을 위해 관련 모듈들을 임포트합니다.

In [ ]:
import asyncio
import time
from typing import AsyncGenerator

## 📌 1단계. Background Task (비동기 배치 작업) 정의
시간이 오래 걸리는 학술 로드맵 합성 연산을 처리하는 백그라운드 작업을 모사합니다.

In [ ]:
async def mock_background_heavy_analysis(task_id: str, paper_count: int):
    print(f"[BackgroundTask] Start analysis for task: {task_id} (Targeting {paper_count} papers)")
    for i in range(1, 4):
        await asyncio.sleep(1) # 1초 대기
        progress = i * 33
        print(f"[BackgroundTask] Task {task_id} Progress: {progress}%...")
    print(f"[BackgroundTask] Task {task_id} COMPLETED successfully! Saving report...")

## 📌 2단계. Server-Sent Events (SSE) 실시간 알림 브로드캐스트
작업 완료 이벤트를 가입한 모든 클라이언트 웹브라우저로 폴링(Polling) 없이 즉각 통보해주는 SSE 브로드캐스터 구조를 학습합니다.

In [ ]:
class MockSSEBroadcaster:
    def __init__(self):
        self.listeners = []
        
    def subscribe(self, listener_queue: asyncio.Queue):
        self.listeners.append(listener_queue)
        print(f"[SSE] Subscriber added. Active connections: {len(self.listeners)}")
        
    def unsubscribe(self, listener_queue: asyncio.Queue):
        if listener_queue in self.listeners:
            self.listeners.remove(listener_queue)
            print(f"[SSE] Subscriber removed. Active connections: {len(self.listeners)}")
            
    async def broadcast(self, event_type: str, data: dict):
        payload = f"event: {event_type}\ndata: {data}\n\n"
        print(f"[SSE] Broadcasting {event_type} event to {len(self.listeners)} clients...")
        for queue in self.listeners:
            await queue.put(payload)

async def sse_event_stream_handler(queue: asyncio.Queue) -> AsyncGenerator[str, None]:
    try:
        while True:
            # 큐에 메시지가 수신될 때까지 양방향 스트리밍 대기
            message = await queue.get()
            yield message
            queue.task_done()
    except asyncio.CancelledError:
        print("[SSE] Stream handler connection cancelled by client disconnect.")

## 📌 3단계. 30분 미활동 보안 세션 및 물리 파일 파쇄 데몬
사용자의 보안 샌드박스 만료 시간을 비교하여, 비활성화 세션을 정리하고 관련 pgvector 테이블을 영구 소거(Wipe Out)하는 백그라운드 스레드 데몬입니다.

In [ ]:
class MockSandboxSession:
    def __init__(self, session_id: str, last_activity_time: float):
        self.session_id = session_id
        self.last_activity_time = last_activity_time

async def wipe_out_expired_sessions_daemon(active_sessions: list[MockSandboxSession], expire_seconds: float):
    print("[CleanupDaemon] Session Cleanup Daemon started.")
    try:
        while True:
            await asyncio.sleep(2) # 2초 주기 검사 실행
            current_time = time.time()
            expired_ids = []
            
            for s in active_sessions[:]:
                elapsed = current_time - s.last_activity_time
                if elapsed > expire_seconds:
                    expired_ids.append(s.session_id)
                    active_sessions.remove(s)
                    
            if expired_ids:
                print(f"[CleanupDaemon] Expired sessions detected: {expired_ids}")
                for sid in expired_ids:
                    print(f"  [WipeOut] Shredding files for session: {sid}")
                    print(f"  [WipeOut] Drop pgvector session table: gem_{sid}_files")
            else:
                print("[CleanupDaemon] All sessions are active and secure.")
(    except asyncio.CancelledError:
        print("[CleanupDaemon] Session Cleanup Daemon shutting down...")

## 📌 4단계. 전체 비동기 파이프라인 통합 시뮬레이션
위의 3가지 핵심 비동기 로직(SSE, BackgroundTasks, 데몬)을 결합하여, 백그라운드 작업 완료 및 스트리밍 알림 발송, 세션 시간 초과 시 자원 소거까지 진행되는 전체 주기를 실행해 봅니다.

In [ ]:
async def run_async_simulation():
    # 1. SSE 브로드캐스터 생성 및 클라이언트 구독
    broadcaster = MockSSEBroadcaster()
    client_queue = asyncio.Queue()
    broadcaster.subscribe(client_queue)
    
    # 2. 세션 리스트 설정 (1개는 만료 직전으로 타임스탬프 설정)
    now = time.time()
    sessions = [
        MockSandboxSession("session_active_101", now),
        MockSandboxSession("session_expired_808", now - 10)  # 10초 전 활동 세션 (만료)
    ]
    
    # 3. 소거 데몬 기동 (만료 기준 5초 설정)
    daemon_task = asyncio.create_task(wipe_out_expired_sessions_daemon(sessions, expire_seconds=5.0))
    
    # 4. 백그라운드 태스크 기동
    task_id = "analysis_task_999"
    bg_task = asyncio.create_task(mock_background_heavy_analysis(task_id, paper_count=3))
    
    # 5. SSE 수신 시뮬레이션
    async def listen_to_sse(generator):
        print("\n--- Client Listening to SSE Stream ---")
        try:
            for _ in range(1):
                message = await anext(generator)
                print(f"\n[Client Received SSE Message]:\n{message.strip()}\n")
        except Exception as e:
            print("SSE Listen failed:", e)
            
    stream_gen = sse_event_stream_handler(client_queue)
    client_task = asyncio.create_task(listen_to_sse(stream_gen))
    
    # 백그라운드 완료 대기 및 알림
    await bg_task
    await broadcaster.broadcast(
        "TASK_COMPLETED", 
        {"task_id": task_id, "status": "success", "message": "Analysis finished!"}
    )
    
    await client_task
    await asyncio.sleep(3)  # 데몬이 만료 세션을 정리할 시간 대기
    
    # 종료 처리
    daemon_task.cancel()
    broadcaster.unsubscribe(client_queue)

await run_async_simulation()